# 第十章：利率互换 / Chapter 10: Interest Rate Swaps

## 你将学到 / What You Will Learn
- 什么是利率互换（用最简单的例子）/ What an interest-rate swap is, explained in the simplest terms
- 为什么两家公司要"交换"利率 / Why two companies would swap rates with each other
- 固定利率 vs 浮动利率的现金流长什么样 / What fixed- vs. floating-rate cash flows look like

---

## 10.1 一个故事 / 10.1 A Quick Story

> **A公司** 从银行借了 1000万，利率是"浮动利率"。
> **B公司** 也借了 1000万，但利率是"固定4%"。
>
> **Company A** borrows 10 M from the bank at a floating rate.
> **Company B** borrows 10 M at a fixed 4 %.
>
> 两家公司商量：
> - A给B **固定4%** 的利息
> - B给A **浮动利率** 的利息
>
> The two firms agree:
> - A pays B **fixed 4 %** interest
> - B pays A the **floating rate** interest
>
> 这样 A锁定了成本，B有机会享受低利率（赌降息）。
>
> A has locked in its cost; B gets to bet on rates falling.

这就是**利率互换(Interest Rate Swap)**！双方交换"固定"和"浮动"的现金流。

That is an **interest-rate swap (IRS)** — the two sides exchange fixed and floating cash flows.

## 10.2 谁赚谁亏？ / 10.2 Who Wins, Who Loses?

- 利率 **涨了** → 浮动利率变大 → B付更多给A → **A赚了** / Rates **rise** → floating goes up → B pays more to A → **A profits**
- 利率 **跌了** → 浮动利率变小 → B付得少 → **B赚了** / Rates **fall** → floating goes down → B pays less → **B profits**

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 10.3 互动实验 / 10.3 Interactive Experiment

> 调整"固定利率"和"当前浮动利率"，观察：
> - 左图：每年双方各付多少钱
> - 中图：固定利率支付方的净损益
> - 右图：利率路径对比
>
> Adjust the fixed rate and the current floating rate and watch:
> - Left chart: how much each side pays each year
> - Middle chart: the fixed-payer's net P&L
> - Right chart: the rate paths side by side

In [ ]:
def interest_rate_swap(notional=1000000, fixed_rate=4.0, init_float_rate=3.5, periods=5):
    plt.close('all')
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    years = np.arange(1, periods + 1)

    np.random.seed(42)
    float_rates = init_float_rate + np.cumsum(np.random.normal(0, 0.3, periods))
    float_rates = np.clip(float_rates, 0.5, 10)

    fixed_cf = np.full(periods, notional * fixed_rate / 100)
    float_cf = notional * float_rates / 100
    net_cf = float_cf - fixed_cf  # from A (fixed-payer) perspective

    # Left
    ax = axes[0]
    w = 0.35
    ax.bar(years - w / 2, fixed_cf / 1000, w, color=C_BLUE, alpha=0.7, label=f'A pays B: fixed {fixed_rate}%')
    ax.bar(years + w / 2, float_cf / 1000, w, color=C_ORANGE, alpha=0.7, label='B pays A: floating')
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Cash flow (k)', fontsize=11)
    ax.set_title('Annual Cash Flows', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xticks(years)
    ax.grid(True, alpha=0.2, axis='y')

    # Middle
    ax = axes[1]
    bar_colors = [C_GREEN if n >= 0 else C_RED for n in net_cf]
    ax.bar(years, net_cf / 1000, color=bar_colors, edgecolor='white', lw=1.5)
    ax.axhline(y=0, color='gray', ls='--')
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Net P&L (k)', fontsize=11)
    ax.set_title('A (fixed-payer) Net P&L', fontsize=13, fontweight='bold')
    ax.set_xticks(years)
    total_net = net_cf.sum()
    sign_color = C_GREEN if total_net >= 0 else C_RED
    ax.text(periods / 2 + 0.5, ax.get_ylim()[1] * 0.8, f'Total: ${total_net / 1000:,.1f}K',
            fontsize=12, fontweight='bold', color=sign_color,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))

    # Right
    ax = axes[2]
    ax.plot(years, float_rates, 'o-', color=C_ORANGE, lw=2.5, ms=8, label='Floating (sim)')
    ax.axhline(y=fixed_rate, color=C_BLUE, lw=2.5, ls='--', label=f'Fixed {fixed_rate}%')
    ax.fill_between(years, float_rates, fixed_rate,
                    where=(float_rates > fixed_rate), color=C_GREEN, alpha=0.2, label='A profits')
    ax.fill_between(years, float_rates, fixed_rate,
                    where=(float_rates <= fixed_rate), color=C_RED, alpha=0.2, label='A loses')
    ax.set_xlabel('Year', fontsize=11)
    ax.set_ylabel('Rate (%)', fontsize=11)
    ax.set_title('Rate Path', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xticks(years)
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    df = pd.DataFrame({
        'Year': years,
        'Fixed (%)': fixed_rate,
        'Floating (%)': [f'{r:.2f}' for r in float_rates],
        'A pays': [f'${fc:,.0f}' for fc in fixed_cf],
        'A receives': [f'${fc:,.0f}' for fc in float_cf],
        'A net': [f'${n:+,.0f}' for n in net_cf],
    })
    display(df)
    print(f'\n  Result: A net P&L over {periods}y = ${total_net:+,.0f}')
    if total_net > 0:
        print(f'  A profited: floating averaged above fixed {fixed_rate}%, locking in fixed was good.')
    else:
        print('  A lost: floating stayed below fixed, locking in fixed cost more.')

interact(interest_rate_swap,
         notional=IntSlider(value=1000000, min=100000, max=10000000, step=100000, description='Notional($):'),
         fixed_rate=FloatSlider(value=4.0, min=1, max=10, step=0.25, description='Fixed(%):'),
         init_float_rate=FloatSlider(value=3.5, min=1, max=8, step=0.25, description='Float now(%):'),
         periods=IntSlider(value=5, min=2, max=10, step=1, description='Years:'));

## 10.4 小结 / 10.4 Chapter Recap

| 概念 / Concept | 一句话 / One-liner |
|------|--------|
| 利率互换 / Interest-rate swap | 两方交换"固定利息"和"浮动利息" / Two parties swap fixed interest for floating interest |
| 固定方 / Fixed payer | 赌利率会涨(锁定成本) / Bets rates will rise (locks in cost) |
| 浮动方 / Floating payer | 赌利率会跌(享受低利率) / Bets rates will fall (enjoys lower rates) |
| 名义本金 / Notional | 计算利息的基数(不会真的交换) / The base for computing interest — never actually exchanged |

> **下一章：永续合约** —— 加密货币世界的独特发明
>
> **Next chapter: perpetual swaps** — a uniquely crypto-native invention.